In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits
from matplotlib.widgets import Slider
from astropy.table import Table
import astropy.units as u
from astropy import constants as const

from reproject import reproject_interp
from scipy.ndimage import fourier_shift
from skimage.registration import phase_cross_correlation
from Functions import *
from astropy.wcs import WCS
from reproject import reproject_interp as rpj
from astropy.convolution import convolve, convolve_fft
from scipy.ndimage import zoom, shift as ndi_shift
from photutils.centroids import centroid_quadratic
import time
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from astropy.wcs.utils import proj_plane_pixel_area

from astropy.visualization import ZScaleInterval
from scipy.optimize import curve_fit


%matplotlib widget



def _gaussian2d_flat(xy, amp, x0, y0, sigma_x, sigma_y, theta, bg):
    """Rotated 2D Gaussian, returns a flat array for curve_fit."""
    xp, yp = xy
    ct, st = np.cos(theta), np.sin(theta)
    xr = (xp - x0) * ct + (yp - y0) * st
    yr = -(xp - x0) * st + (yp - y0) * ct
    return (bg + amp * np.exp(
        -0.5 * (xr**2 / sigma_x**2 + yr**2 / sigma_y**2)
    )).ravel()

def coarse_centroid(image, x0, y0, search_box=6,
                    threshold_sigma=2.0, max_iter=10):
    """
    Iterative intensity-weighted centroid on a small stamp.
    Returns integer (cx, cy) within a few pixels of the source peak.
    """
    ny, nx = image.shape
    cx, cy = int(round(x0)), int(round(y0))

    for _ in range(max_iter):
        h  = search_box // 2
        x1 = max(0, cx - h);  x2 = min(nx, cx + h + 1)
        y1 = max(0, cy - h);  y2 = min(ny, cy + h + 1)

        stamp = image[y1:y2, x1:x2].copy()
        bg    = np.median(stamp)
        stamp -= bg

        mad   = np.median(np.abs(stamp))
        stamp = np.where(stamp > threshold_sigma * 1.4826 * mad, stamp, 0.0)

        total = stamp.sum()
        if total <= 0:
            break

        yy, xx   = np.indices(stamp.shape)
        cx_new   = int(round(x1 + (xx * stamp).sum() / total))
        cy_new   = int(round(y1 + (yy * stamp).sum() / total))
        cx_new   = max(0, min(nx - 1, cx_new))
        cy_new   = max(0, min(ny - 1, cy_new))

        if cx_new == cx and cy_new == cy:
            break
        cx, cy = cx_new, cy_new

    return cx, cy

def fit_2d_gaussian(cutout, fit_radius=15):
    """
    Fit a rotated 2D Gaussian to pixels within fit_radius of the
    cutout centre.

    Returns
    -------
    result : dict with keys
        fwhm_x, fwhm_y  – along the Gaussian principal axes (px)
        fwhm_mean       – geometric-mean FWHM = 2.3548 * sqrt(sx*sy)
        fwhm_major      – larger of fwhm_x, fwhm_y
        fwhm_minor      – smaller
        theta           – position angle of major axis (radians)
        sub_x, sub_y    – sub-pixel centre offset from cutout centre
        popt            – full parameter vector [amp,x0,y0,sx,sy,theta,bg]
    """
    ny, nx = cutout.shape
    cy, cx = ny // 2, nx // 2

    y_idx, x_idx = np.indices(cutout.shape)
    r = np.sqrt((x_idx - cx)**2 + (y_idx - cy)**2)

    # Background from outer annulus
    rmax    = min(cx, cy)
    outer   = r >= max(rmax - 5, rmax * 0.8)
    bg_est  = float(np.median(cutout[outer])) if outer.any() else 0.0

    # Pixels used for fitting
    mask = r <= fit_radius
    xd   = x_idx[mask].ravel().astype(float)
    yd   = y_idx[mask].ravel().astype(float)
    zd   = cutout[mask].ravel().astype(float)

    amp_est = float(np.max(zd) - bg_est)
    if amp_est <= 0:
        raise ValueError("No positive signal in fit window")

    p0 = [amp_est, float(cx), float(cy), 2.0, 2.0, 0.0, bg_est]
    lo = [0,        cx - fit_radius, cy - fit_radius,
          0.3,  0.3,  -np.pi / 2, -np.inf]
    hi = [np.inf,   cx + fit_radius, cy + fit_radius,
          fit_radius, fit_radius, np.pi / 2,  np.inf]

    popt, _ = curve_fit(
        _gaussian2d_flat, (xd, yd), zd,
        p0=p0, bounds=(lo, hi), maxfev=20000,
    )

    amp, x0f, y0f, sx, sy, theta, bg = popt
    sx, sy = abs(sx), abs(sy)

    fwhm_x = 2.3548 * sx
    fwhm_y = 2.3548 * sy
    fwhm_mean  = 2.3548 * np.sqrt(sx * sy)
    fwhm_major = max(fwhm_x, fwhm_y)
    fwhm_minor = min(fwhm_x, fwhm_y)

    return dict(
        fwhm_x=fwhm_x, fwhm_y=fwhm_y,
        fwhm_mean=fwhm_mean,
        fwhm_major=fwhm_major, fwhm_minor=fwhm_minor,
        theta=theta,
        sub_x=x0f - cx, sub_y=y0f - cy,
        popt=popt,
    )

def percentile_stretch(image, lo=1, hi=99.5):
    data = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
    vmin = np.percentile(data, lo)
    vmax = np.percentile(data, hi)
    return data, vmin, max(vmax, vmin + 1)

def _binned_radial_profile(cutout, cx_sub, cy_sub, fit_radius):
    """Binned radial profile centred on the sub-pixel fit centre."""
    ny, nx = cutout.shape
    y_idx, x_idx = np.indices(cutout.shape)
    r = np.sqrt((x_idx - cx_sub)**2 + (y_idx - cy_sub)**2)
    r_int = r.astype(int)
    rmax  = int(np.floor(fit_radius)) + 1
    profile = np.full(rmax, np.nan)
    for i in range(rmax):
        m = r_int == i
        if m.any():
            profile[i] = np.mean(cutout[m])
    valid = ~np.isnan(profile)
    return np.arange(rmax)[valid], profile[valid]

def process_click(
    x0, y0,
    full_image,
    half, fit_radius,
    nx_full, ny_full,
    ax_cutout, ax_profile,
    ov_marker,           # matplotlib artist on overview panel
    zscale, fig, search_box=6
):
    """
    1. Coarse centroid to snap onto nearest source (integer px).
    2. Extract cutout centred on that integer position.
    3. Fit a 2D Gaussian with free sub-pixel centre.
    4. Update all three display panels.

    Returns (cutout, cx, cy) on success, None on failure.
    """
    # --- guard edges -----------------------------------------
    if (x0 < half or y0 < half
            or x0 > nx_full - half - 1
            or y0 > ny_full - half - 1):
        print("Click too close to image edge – ignored.")
        return None

    # --- coarse centroid -------------------------------------
    cx, cy = coarse_centroid(full_image, x0, y0,search_box=search_box)
    print(f"  Coarse centroid → ({cx}, {cy})")

    if (cx < half or cy < half
            or cx > nx_full - half - 1
            or cy > ny_full - half - 1):
        print("  Centroid too close to edge – ignored.")
        return None

    # --- cutout (centred on integer centroid) ----------------
    cutout = full_image[
        cy - half: cy + half + 1,
        cx - half: cx + half + 1,
    ].copy()

    # --- 2D Gaussian fit -------------------------------------
    try:
        res = fit_2d_gaussian(cutout, fit_radius=fit_radius)
        fit_ok = True
    except Exception as e:
        print(f"  2D fit failed: {e}")
        fit_ok = False
        res = {}

    # --- update overview marker ------------------------------
    if ov_marker is not None:
        ov_marker.set_data([cx], [cy])

    # --- cutout panel ----------------------------------------
    ax_cutout.clear()
    try:
        vc_min, vc_max = zscale.get_limits(cutout)
    except Exception:
        _, vc_min, vc_max = percentile_stretch(cutout)

    ax_cutout.imshow(
        cutout, origin='lower', cmap='inferno',
        vmin=vc_min, vmax=vc_max, interpolation='nearest',
    )

    if fit_ok:
        # Mark the sub-pixel fit centre
        fx = half + res['sub_x']
        fy = half + res['sub_y']
        ax_cutout.plot(fx, fy, '+', color='cyan', ms=5, mew=0.5)

        # Draw the FWHM ellipse
        from matplotlib.patches import Ellipse
        ellipse = Ellipse(
            xy=(fx, fy),
            width=res['fwhm_x'], height=res['fwhm_y'],
            angle=np.degrees(res['theta']),
            edgecolor='lime', facecolor='none', lw=0.5, alpha=0.5,
        )
        ax_cutout.add_patch(ellipse)
    else:
        ax_cutout.axhline(half, color='cyan', lw=0.4, alpha=0.5)
        ax_cutout.axvline(half, color='cyan', lw=0.4, alpha=0.5)

    ax_cutout.set_title(
        f"Cutout  (x={cx}, y={cy})\nClick here to refine", fontsize=9
    )

    # --- profile panel ---------------------------------------
    ax_profile.clear()

    if fit_ok:
        # Background-subtracted cutout for profile
        bg_val = res['popt'][6]
        cutout_bs = cutout - bg_val

        r_bins, prof = _binned_radial_profile(
            cutout_bs,
            half + res['sub_x'],
            half + res['sub_y'],
            fit_radius,
        )

        # Normalise
        amp = res['popt'][0]
        r_fine = np.linspace(0, fit_radius, 400)
        gauss1d = amp * np.exp(-0.5 * r_fine**2 /
                               (res['popt'][3] * res['popt'][4]))  # geometric sigma

        ax_profile.scatter(r_bins, prof / amp, s=18,
                           color='steelblue', zorder=3, label='Radial profile')
        ax_profile.plot(r_fine, gauss1d / amp, '-', lw=2, color='tomato',
                        label=(f"2D Gaussian fit\n"
                               f"FWHM_max = {res['fwhm_major']:.2f} px\n"
                               f"FWHM_min = {res['fwhm_minor']:.2f} px\n"
                               f"FWHM_mean = {res['fwhm_mean']:.2f} px"))
        ax_profile.axhline(0.5, color='gray', lw=0.8, ls='--', alpha=0.6)
        title = (f"FWHM  maj={res['fwhm_major']:.2f}"
                 f"min={res['fwhm_minor']:.2f}  "
                 f"mean={res['fwhm_mean']:.2f} px")
    else:
        title = "Fit failed"

    ax_profile.set_xlabel("Radius (pixels)", fontsize=9)
    ax_profile.set_ylabel("Normalised Intensity", fontsize=9)
    ax_profile.set_title(title, fontsize=9)
    if fit_ok:
        ax_profile.legend(fontsize=8)
    ax_profile.set_ylim(-0.15, 1.25)

    fig.canvas.draw_idle()

    eccentricity = 1-(res['fwhm_minor']/res['fwhm_major'])
    
    if fit_ok:
        print(f"  x={cx}, y={cy}  |  "
              f"FWHM_max={res['fwhm_major']:.3f}  "
              f"FWHM_min={res['fwhm_minor']:.3f}  "
              f"FWHM_mean={res['fwhm_mean']:.3f} px  "
              f"Eccentricity={eccentricity:.3f}   "
              f"sub-pixel offset=({res['sub_x']:+.2f}, {res['sub_y']:+.2f})")
    
    return cutout, cx, cy

def interactive_fwhm_inspector(
    image_file,
    cutout_size=101,
    fit_radius=15,
    overview_size=1500,
):
    """
    Two-level interactive FWHM inspector.

    Left panel   – downsampled overview; click to navigate.
    Centre panel – full-resolution cutout with FWHM ellipse;
                   click to refine (re-centroid + re-fit).
    Right panel  – binned radial profile + 2D Gaussian fit.

    Parameters
    ----------
    image_file   : path to FITS file
    cutout_size  : side length of analysis cutout in pixels (odd)
    fit_radius   : radius in pixels used for the 2D Gaussian fit
    overview_size: longest axis of the downsampled overview (px)
    """
    print("Loading FITS …")
    full_image = fits.getdata(image_file).astype(float)
    full_image = np.nan_to_num(full_image, nan=0.0, posinf=0.0, neginf=0.0)
    ny_full, nx_full = full_image.shape

    factor   = max(1, max(ny_full, nx_full) // overview_size)
    overview = full_image[::factor, ::factor]

    print(f"Full image : {nx_full}×{ny_full}")
    print(f"Overview   : {overview.shape[1]}×{overview.shape[0]}  (1/{factor})")

    half = cutout_size // 2

    zscale = ZScaleInterval(n_samples=10000, contrast=0.25)
    try:
        vmin_ov, vmax_ov = zscale.get_limits(overview)
    except Exception:
        _, vmin_ov, vmax_ov = percentile_stretch(overview)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    ax_image, ax_cutout, ax_profile = axes

    ax_image.imshow(
        overview, origin='lower', cmap='gray',
        vmin=vmin_ov, vmax=vmax_ov, interpolation='nearest',
    )
    ax_image.set_title("Overview — click to select source\n(ZScale stretch)", fontsize=9)

    # Overview marker shown in full-image pixel coords
    ov_marker, = ax_image.plot([], [], 'r+', ms=14, mew=1.5,
                               transform=ax_image.transData)

    # The overview axes are in downsampled coords, so we need a wrapper
    # that converts the marker position from full-image → overview coords.
    class _OvMarker:
        def set_data(self, xs, ys):
            ov_marker.set_data(
                [x / factor for x in xs],
                [y / factor for y in ys],
            )

    ov_marker_wrapped = _OvMarker()

    ax_cutout.set_title("Cutout", fontsize=9)
    ax_profile.set_title("Radial Profile", fontsize=9)

    state = {'cutout': None, 'x0': None, 'y0': None}

    # --------------------------------------------------------
    # Overview click → navigate
    # --------------------------------------------------------
    def onclick_overview(event):
        if event.inaxes != ax_image:
            return
        x0 = int(round(event.xdata)) * factor
        y0 = int(round(event.ydata)) * factor
        result = process_click(
            x0, y0, full_image, half, fit_radius, nx_full, ny_full,
            ax_cutout, ax_profile, ov_marker_wrapped, zscale, fig,
        )
        if result is not None:
            state['cutout'], state['x0'], state['y0'] = result
            fig.canvas.draw_idle()

    # --------------------------------------------------------
    # Cutout click → refine
    # --------------------------------------------------------
    def onclick_cutout(event):
        if event.inaxes != ax_cutout or state['cutout'] is None:
            return
        dx = int(round(event.xdata)) - half
        dy = int(round(event.ydata)) - half
        x0_new = state['x0'] + dx
        y0_new = state['y0'] + dy
        result = process_click(
            x0_new, y0_new, full_image, half, fit_radius, nx_full, ny_full,
            ax_cutout, ax_profile, ov_marker_wrapped, zscale, fig,
        )
        if result is not None:
            state['cutout'], state['x0'], state['y0'] = result
            fig.canvas.draw_idle()

    fig.canvas.mpl_connect('button_press_event', onclick_overview)
    fig.canvas.mpl_connect('button_press_event', onclick_cutout)

    plt.tight_layout()
    plt.show()


In [ ]:
fits.open('/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits').info()

In [ ]:
from obszugang.phot_access import PhotAccess as PA

access = PA(phot_hst_target_name= 'ngc1672')
access.load_hst_band(band='F658N', load_err=True)
access.hst_bands_data.keys()

In [ ]:
ngc1672 = ImageScience()
ngc1672.load_image('f187', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f187n_i2d_anchor.fits')
ngc1672.load_image('f150', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f150w_i2d_anchor_aligned_convolved.fits')
ngc1672.load_image('f658', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/acsf658n/ngc1672_acs_f658n_exp_drc_sci.fits')
ngc1672.load_image('f555', '/project/galaxies/tjuchau/data_files/HST/ngc1672/hlsp_phangs-hst_hst_wfc3-uvis_ngc1672mosaic_f555w_v1_exp-drc-sci.fits')
ngc1672.sub_images('f658', 'f555', scales=[1,1.271], out_name='my_cont')
ngc1672.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTSUB")
ngc1672.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTINUUM")
ngc1672.sum_images('pa_cont', 'pa_contsub', out_name='pa_full')
ngc1672.load_image('ha_contsub', '/project/galaxies/tjuchau/data_files/HST/ngc1672/ngc1672_hst_ha_contsub_aligned.fits')
ngc1672.load_image('ha_cont', '/project/galaxies/tjuchau/data_files/HST/ngc1672/ngc1672_hst_ha_continuum.fits')
ngc1672.sub_images('ha_cont', 'ha_contsub', out_name='ha_full')
print(f'Loaded galaxy ngc1672 with callable keys :\n{list(ngc1672.images.keys())}')

In [ ]:
interactive_fwhm_inspector(
    ngc1672.files['pa_contsub'],
    cutout_size=500,
    fit_radius=15,
    overview_size=1500,
)

In [ ]:
ngc1672.inspect_continuum_subtraction('pa_cont', 'pa_full', initial_scale=1, zoom_size=600, zoom_center=(925, 3088))

In [ ]:
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table = table[table['galaxy']=="ngc1672"]
table[0]
pa_EW = []
ha_EW = []
for row in table:
    pa_EW.append(ngc1672.get_equivalent_width('pa_full', 'pa_cont', [row['ra'], row['dec']], row['radius']*u.arcsec, 0.1*u.arcsec)[0].value)
    print(pa_EW[-1])
    ha_EW.append(ngc1672.get_equivalent_width('f658', 'ha_cont', [row['ra'], row['dec']], row['radius']*u.arcsec, 0.1*u.arcsec)[0].value)
    print(ha_EW[-1])

In [ ]:
import obszugang
from obszugang import cluster_cat_access
x = cluster_cat_access.ClusterCatAccess()
ra,dec = x.get_hst_cc_coords_world(target = 'ngc1672')
i += 1
ngc1672.display(['pa_contsub','pa_cont', 'f555', 'f658'], [ra[i],dec[i]], 0.15*u.arcsec, ncols=2, zoom=10)

In [ ]:
ngc1672.get_equivalent_width('f658', 'f555', [ra[i], dec[i]], 0.15*u.arcsec, 0.05*u.arcsec)

In [ ]:
table=Table(names=['ra', 'dec', 'pa_EW', 'ha_EW'])
for r,d in zip(ra,dec):
    try:
        pa_EW = ngc1672.get_equivalent_width('pa_full', 'pa_cont', [r, d], 0.15*u.arcsec, 0.05*u.arcsec)[0].value
        ha_EW = ngc1672.get_equivalent_width('f658', 'f555', [r, d], 0.15*u.arcsec, 0.05*u.arcsec)[0].value
        table.add_row([r,d,pa_EW,ha_EW])
    except:
        table.add_row([r,d,0,0])

plt.clf()
plt.scatter(table['pa_EW'], table['ha_EW'])
plt.xlabel('pa_EW')
plt.ylabel('ha_EW')
plt.show()

In [ ]:
plt.clf()
plt.scatter(table['pa_EW'], table['ha_EW'])
plt.xlabel('pa_EW')
plt.ylabel('ha_EW')
plt.show()

In [ ]:
table[table['pa_EW']>10000]

In [ ]:
ngc1672.display(['pa_contsub','pa_cont', 'f555', 'f658'], [71.42643466680812,-59.24848747089673], 0.15*u.arcsec, ncols=2, zoom=50)